In [1]:
!pip install wandb

In [2]:
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
#from google.colab import drive

wandb.login(key="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")

Matplotlib is building the font cache; this may take a moment.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/andrescubillovillalobos/.netrc
wandb: Currently logged in as: andrescubillo07 (andrescubillo07-tecnol-gico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statistics import mean

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.preprocessing import robust_scale,minmax_scale
from sklearn.model_selection import KFold

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import L1L2, L1, L2
from tensorflow.keras.optimizers import Adam

In [4]:
data = pd.read_csv("./../Data_Demo/filtered_input_train_lnc_features.csv", header=None)

In [5]:
data

,0,1,2,3,4,5,6,7,8,9,...,681,682,683,684,685,686,687,688,689,690
0,82,2763,825.0,52.0,25.0,-116090.0,18.0,0.0,4.0,-1250.0,...,4.0,11.0,6.0,9.0,7.0,12.0,8.0,10.0,9.0,15.0
1,82,1127,308.0,23.0,9.0,-33410.0,3.0,1.0,3.0,570.0,...,0.0,8.0,4.0,6.0,9.0,4.0,5.0,9.0,9.0,30.0
2,82,1081,295.0,21.0,11.0,-33570.0,3.0,0.0,3.0,840.0,...,0.0,7.0,4.0,5.0,8.0,5.0,5.0,10.0,9.0,29.0
3,82,1202,325.0,27.0,14.0,-37340.0,3.0,0.0,2.0,140.0,...,0.0,8.0,5.0,5.0,8.0,6.0,6.0,11.0,9.0,32.0
4,82,1085,292.0,24.0,11.0,-33610.0,3.0,0.0,2.0,140.0,...,0.0,7.0,4.0,5.0,8.0,5.0,5.0,10.0,9.0,29.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,82,873,263.0,19.0,7.0,-28490.0,-1.0,-1.0,-1.0,0.0,...,1.0,2.0,5.0,3.0,8.0,2.0,3.0,5.0,5.0,7.0
886,71,473,126.0,13.0,2.0,-16200.0,-1.0,-1.0,-1.0,0.0,...,0.0,2.0,1.0,1.0,4.0,2.0,2.0,1.0,2.0,3.0
887,84,3201,917.0,57.0,25.0,-95510.0,13.0,0.0,3.0,-1390.0,...,2.0,22.0,15.0,10.0,17.0,31.0,31.0,26.0,32.0,102.0
888,82,8797,2702.0,139.0,59.0,-274977.0,13.0,1.0,9.0,-950.0,...,1.0,64.0,57.0,37.0,45.0,49.0,93.0,84.0,79.0,174.0


In [6]:
labels = [1] * 528 + [0] * 362
labels = pd.DataFrame(labels)

In [7]:
train_data, test_data, train_label, test_label = train_test_split(data, labels, test_size=0.25, stratify=labels, random_state=40)
dim = len(train_data.columns)

In [8]:
k =  5
kf = KFold(n_splits=k,shuffle=True,random_state=42)

In [9]:
def get_model(do):
    model = Sequential()
    model.add(BatchNormalization(input_shape = (dim,)))
    model.add(Dense(150,activation = 'relu', bias_initializer='zeros', kernel_initializer = 'he_normal', kernel_regularizer=L2(0.002)))
    model.add(Dropout(do))
    model.add(Dense(75,activation = 'relu', bias_initializer='zeros', kernel_initializer = 'he_normal', kernel_regularizer=L2(0.002)))
    model.add(Dropout(do))
    model.add(Dense(20,activation = 'relu', bias_initializer='zeros', kernel_initializer = 'he_normal', kernel_regularizer=L2(0.002)))
    model.add(Dropout(do))
    model.add(Dense(1,activation = 'sigmoid', bias_initializer = 'zeros', kernel_initializer = 'glorot_normal'))
    return model

In [10]:
sweep_config = {
    'method': 'grid',
    "name": "sweep-state-40",
    "parameters":
      {
        "batch_size": {"values": [4, 8, 12]},
        "epochs": {"values": [100, 150]},
        'dropout': {'values': [0.2, 0.3, 0.4]},
        'lr': {'values': [0.001, 0.002, 0.0001, 0.0002]}
      }
}

def sweep_train(config_defaults=None):
    with wandb.init(config=config_defaults) as run:
        run.name = f'b{wandb.config.batch_size}_ep{wandb.config.epochs}_dp{wandb.config.dropout}_lr{wandb.config.lr}'
        model = get_model(wandb.config.dropout)
        opt = Adam(wandb.config.lr)
        model.compile(optimizer=opt,loss='binary_crossentropy', metrics=['accuracy'])

        wandb_callbacks = [
          WandbMetricsLogger()
        ]

        model.fit(train_data, train_label, batch_size = wandb.config.batch_size, epochs = wandb.config.epochs, validation_data = (test_data, test_label), callbacks = wandb_callbacks)

sweep_id = wandb.sweep(sweep_config, project="train-lncfeatures-filtereddata")
wandb.agent(sweep_id, function=sweep_train)


Create sweep with ID: ermtwovc
Sweep URL: https://wandb.ai/andrescubillo07-tecnol-gico-de-costa-rica/train-lncfeatures-filtereddata/sweeps/ermtwovc


wandb: Agent Starting Run: fppfc22r with config:
wandb: 	batch_size: 4
wandb: 	dropout: 0.2
wandb: 	epochs: 100
wandb: 	lr: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/andrescubillovillalobos/.netrc.


Epoch 1/100


/opt/homebrew/Caskroom/miniforge/base/envs/compgen/lib/python3.11/site-packages/keras/src/layers/normalization/batch_normalization.py:181: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5877 - loss: 1.5873 - val_accuracy: 0.7399 - val_loss: 1.3786
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 873us/step - accuracy: 0.6567 - loss: 1.4041 - val_accuracy: 0.6951 - val_loss: 1.2715
Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 885us/step - accuracy: 0.6447 - loss: 1.3227 - val_accuracy: 0.7265 - val_loss: 1.2405
Epoch 4/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 954us/step - accuracy: 0.7076 - loss: 1.2415 - val_accuracy: 0.8520 - val_loss: 1.0807
Epoch 5/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 903us/step - accuracy: 0.7316 - loss: 1.1370 - val_accuracy: 0.7982 - val_loss: 0.9866
Epoch 6/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 904us/step - accuracy: 0.7586 - loss: 1.0630 - val_accuracy: 0.8610 - val_loss: 0.9355
Epoch 7/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 874us/step - accuracy: 0.7826 - loss: 0.9807 - val_accuracy: 0.8027 - val_loss: 0.9025
Epoch 8/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 886us/step - accuracy: 0.7976 - loss: 0.9284 - val_ac

...

Epoch 1/150


/opt/homebrew/Caskroom/miniforge/base/envs/compgen/lib/python3.11/site-packages/keras/src/layers/normalization/batch_normalization.py:181: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5052 - loss: 2.1045 - val_accuracy: 0.5874 - val_loss: 1.6742
Epoch 2/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5847 - loss: 1.8152 - val_accuracy: 0.6323 - val_loss: 1.5914
Epoch 3/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5637 - loss: 1.7551 - val_accuracy: 0.6637 - val_loss: 1.5244
Epoch 4/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5712 - loss: 1.6706 - val_accuracy: 0.6682 - val_loss: 1.5017
Epoch 5/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5892 - loss: 1.6281 - val_accuracy: 0.6906 - val_loss: 1.4771
Epoch 6/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6177 - loss: 1.5459 - val_accuracy: 0.7265 - val_loss: 1.4540
Epoch 7/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5952 - loss: 1.5895 - val_accuracy: 0.7578 - val_loss: 1.4483
Epoch 8/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6582 - loss: 1.4908 - val_accuracy: 0.7085 - val_loss: 1.4

epoch/accuracy,▁▂▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇█▇▇█▇██████
epoch/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▇▆▆▆▅▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▃▂▃▄▅▆▆▇▇▆▆▇▇▇█▇▇▇▇▇▇█▇█▇▇▇████▇█████
epoch/val_loss,███▇▇▆▆▆▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▂▁▁▁
epoch/accuracy,0.92354
epoch/epoch,149
epoch/learning_rate,0.0002
epoch/loss,0.51761
epoch/val_accuracy,0.93722


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.
